# IELTS Chatbot Standalone on Colab

Run the cells from top to bottom. The notebook starts Ollama, the FastAPI backend, warms up the large models, then starts the Vite frontend and Cloudflare Tunnel.

Runtime pieces:

- Ollama + `hf.co/Zkare/Chatbot_Ielts_Assistant_v2:Q4_K_M`
- Document RAG for TXT/Markdown, PDF, DOCX, and images
- RapidOCR PP-OCRv6 medium with PyTorch CUDA as the only OCR path; no fallback OCR model
- FastAPI backend on port 2222
- Vite/React frontend on port 8000


In [1]:
REPO_URL = "https://github.com/phamtu2x5/ielts-chatbot"
REPO_DIR = "/content/ielts-chatbot"

OLLAMA_MODEL = "hf.co/Zkare/Chatbot_Ielts_Assistant_v2:Q4_K_M"
OLLAMA_API_URL = "http://127.0.0.1:11434/api/generate"
OLLAMA_NUM_PREDICT = 1200
OLLAMA_NUM_CTX = 4096
OLLAMA_TIMEOUT_SECONDS = 180
OLLAMA_THINK = False

BACKEND_PORT = 2222
FRONTEND_PORT = 8000

# Keep the same embedding model as the backend and persisted index.
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
RAG_TOP_K = 5
RAG_MIN_SCORE = 0.45
RAG_PROBE_TOP_K = 3
RAG_PROBE_MIN_DENSE_SCORE = 0.35
RAG_OVERVIEW_TOP_K = 8
RAG_OVERVIEW_SOURCE_CHARS = 900

# Keep ingestion bounded for a temporary Colab demo.
DOCUMENT_MAX_UPLOAD_MB = 25
DOCUMENT_MAX_PDF_PAGES = 80
DOCUMENT_CHUNK_TARGET_TOKENS = 600
DOCUMENT_CHUNK_MAX_TOKENS = 800
DOCUMENT_CHUNK_OVERLAP_TOKENS = 80
DOCUMENT_ENABLE_IELTS_STRUCTURE = True
DOCUMENT_OCR_DUPLICATE_SIMILARITY = 0.88
DOCUMENT_OCR_DUPLICATE_TOKEN_OVERLAP = 0.92
DOCUMENT_OCR_MIN_NEW_TOKEN_RATIO = 0.08
DOCUMENT_OCR_DPI = 180

# OCR is native-first and uses one RapidOCR/PyTorch CUDA model when OCR is needed.
OCR_ENGINE = "rapidocr"
OCR_RUNTIME = "torch"
OCR_DEVICE = "cuda:0"
OCR_LANG = "en"
OCR_DET_LANG = "ch"
OCR_VERSION = "PP-OCRv6"
OCR_MODEL_SIZE = "medium"
OCR_MIN_CONFIDENCE = 0.72

# DocLayout-YOLO detects visual regions such as tables and figures before OCR/table parsing.
LAYOUT_ENABLE = True
LAYOUT_ENGINE = "doclayout_yolo"
LAYOUT_DEVICE = "cuda:0"
LAYOUT_MODEL_REPO = "juliozhao/DocLayout-YOLO-DocStructBench"
LAYOUT_MODEL_FILENAME = "doclayout_yolo_docstructbench_imgsz1024.pt"
LAYOUT_MODEL_PATH = ""
LAYOUT_CONFIDENCE = 0.25
LAYOUT_IMAGE_SIZE = 1024

# Warm up all large models before opening the public UI.
WARMUP_LLM = True
WARMUP_EMBEDDING = True
WARMUP_OCR = True
WARMUP_LAYOUT = True

print("Repo:", REPO_URL)
print("LLM:", OLLAMA_MODEL)
print("Embedding:", EMBEDDING_MODEL_NAME)
print("OCR:", OCR_ENGINE, OCR_RUNTIME, OCR_VERSION, OCR_MODEL_SIZE)
print("Layout:", LAYOUT_ENGINE, LAYOUT_MODEL_REPO, "enabled=", LAYOUT_ENABLE)


In [3]:
!apt-get update -y
!apt-get install -y git git-lfs curl zstd
!git lfs install


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [100 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,820 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.5 MB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jamm

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
import os
import subprocess
import time
import requests
from pathlib import Path

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

def is_ollama_ready():
    try:
        response = requests.get("http://127.0.0.1:11434/api/tags", timeout=3)
        return response.status_code == 200
    except Exception:
        return False

if not is_ollama_ready():
    subprocess.run(["pkill", "-f", "ollama serve"], check=False)
    ollama_log = open("/content/ollama.log", "w")
    ollama_proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    print("Started Ollama PID:", ollama_proc.pid)
else:
    print("Ollama is already running")

for _ in range(45):
    if is_ollama_ready():
        print("Ollama ready")
        break
    time.sleep(1)
else:
    print(Path("/content/ollama.log").read_text()[-4000:])
    raise RuntimeError("Ollama did not become ready")

print(requests.get("http://127.0.0.1:11434/api/tags", timeout=10).text)


Started Ollama PID: 1756
Ollama ready
{"models":[]}


In [6]:
!ollama pull {OLLAMA_MODEL}
!ollama list



NAME                                             ID              SIZE      MODIFIED               
hf.co/Zkare/Chatbot_Ielts_Assistant_v2:Q4_K_M    ce8844ecb130    2.5 GB    Less than a second ago    


In [7]:
!rm -rf {REPO_DIR}
!git clone --depth 1 {REPO_URL} {REPO_DIR}
!cd {REPO_DIR} && git lfs pull


Cloning into '/content/ielts-chatbot'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 51 (delta 1), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 100.75 KiB | 838.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.


In [8]:
import importlib.metadata as metadata
import subprocess
import sys


def run(command):
    print("$", " ".join(str(part) for part in command))
    subprocess.check_call([str(part) for part in command])


def installed_version(package_name):
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return None


run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade-strategy",
    "only-if-needed",
    "-r",
    f"{REPO_DIR}/backend/requirements.txt",
])

import rapidocr
import doclayout_yolo
import torch

print("RapidOCR version:", installed_version("rapidocr"))
print("DocLayout-YOLO version:", installed_version("doclayout-yolo"))
print("Torch version:", torch.__version__)
print("Torch CUDA available:", torch.cuda.is_available())



In [9]:
import os
import subprocess
import time
import socket
import requests
from pathlib import Path

previous_backend = globals().get("backend_proc")
if previous_backend is not None and previous_backend.poll() is None:
    previous_backend.terminate()
    try:
        previous_backend.wait(timeout=15)
    except subprocess.TimeoutExpired:
        previous_backend.kill()
        previous_backend.wait(timeout=5)

previous_backend_log = globals().get("backend_log")
if previous_backend_log is not None and not previous_backend_log.closed:
    previous_backend_log.close()

# Stop only a stale Uvicorn process serving this app on the configured port.
subprocess.run(
    ["pkill", "-TERM", "-f", f"uvicorn app.main:app.*--port {BACKEND_PORT}"],
    check=False,
)

def backend_port_is_open():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex(("127.0.0.1", BACKEND_PORT)) == 0

for _ in range(15):
    if not backend_port_is_open():
        break
    time.sleep(1)
else:
    # Colab provides fuser; use it only for the configured backend port.
    subprocess.run(["fuser", "-k", f"{BACKEND_PORT}/tcp"], check=False)
    time.sleep(2)
    if backend_port_is_open():
        raise RuntimeError(f"Port {BACKEND_PORT} is still occupied; restart the Colab runtime.")

backend_env = os.environ.copy()
backend_env.update({
    "OLLAMA_API_URL": OLLAMA_API_URL,
    "OLLAMA_MODEL": OLLAMA_MODEL,
    "OLLAMA_NUM_PREDICT": str(OLLAMA_NUM_PREDICT),
    "OLLAMA_NUM_CTX": str(OLLAMA_NUM_CTX),
    "OLLAMA_TIMEOUT_SECONDS": str(OLLAMA_TIMEOUT_SECONDS),
    "OLLAMA_THINK": str(OLLAMA_THINK).lower(),
    "EMBEDDING_MODEL_NAME": EMBEDDING_MODEL_NAME,
    "RAG_TOP_K": str(RAG_TOP_K),
    "RAG_MIN_SCORE": str(RAG_MIN_SCORE),
    "RAG_PROBE_TOP_K": str(RAG_PROBE_TOP_K),
    "RAG_PROBE_MIN_DENSE_SCORE": str(RAG_PROBE_MIN_DENSE_SCORE),
    "RAG_OVERVIEW_TOP_K": str(RAG_OVERVIEW_TOP_K),
    "RAG_OVERVIEW_SOURCE_CHARS": str(RAG_OVERVIEW_SOURCE_CHARS),
    "DOCUMENT_MAX_UPLOAD_MB": str(DOCUMENT_MAX_UPLOAD_MB),
    "DOCUMENT_MAX_PDF_PAGES": str(DOCUMENT_MAX_PDF_PAGES),
    "DOCUMENT_CHUNK_TARGET_TOKENS": str(DOCUMENT_CHUNK_TARGET_TOKENS),
    "DOCUMENT_CHUNK_MAX_TOKENS": str(DOCUMENT_CHUNK_MAX_TOKENS),
    "DOCUMENT_CHUNK_OVERLAP_TOKENS": str(DOCUMENT_CHUNK_OVERLAP_TOKENS),
    "DOCUMENT_ENABLE_IELTS_STRUCTURE": str(DOCUMENT_ENABLE_IELTS_STRUCTURE).lower(),
    "DOCUMENT_OCR_DUPLICATE_SIMILARITY": str(DOCUMENT_OCR_DUPLICATE_SIMILARITY),
    "DOCUMENT_OCR_DUPLICATE_TOKEN_OVERLAP": str(DOCUMENT_OCR_DUPLICATE_TOKEN_OVERLAP),
    "DOCUMENT_OCR_MIN_NEW_TOKEN_RATIO": str(DOCUMENT_OCR_MIN_NEW_TOKEN_RATIO),
    "DOCUMENT_OCR_DPI": str(DOCUMENT_OCR_DPI),
    "OCR_ENGINE": OCR_ENGINE,
    "OCR_RUNTIME": OCR_RUNTIME,
    "OCR_DEVICE": OCR_DEVICE,
    "OCR_LANG": OCR_LANG,
    "OCR_DET_LANG": OCR_DET_LANG,
    "OCR_VERSION": OCR_VERSION,
    "OCR_MODEL_SIZE": OCR_MODEL_SIZE,
    "OCR_MIN_CONFIDENCE": str(OCR_MIN_CONFIDENCE),
    "LAYOUT_ENABLE": str(LAYOUT_ENABLE).lower(),
    "LAYOUT_ENGINE": LAYOUT_ENGINE,
    "LAYOUT_DEVICE": LAYOUT_DEVICE,
    "LAYOUT_MODEL_REPO": LAYOUT_MODEL_REPO,
    "LAYOUT_MODEL_FILENAME": LAYOUT_MODEL_FILENAME,
    "LAYOUT_MODEL_PATH": LAYOUT_MODEL_PATH,
    "LAYOUT_CONFIDENCE": str(LAYOUT_CONFIDENCE),
    "LAYOUT_IMAGE_SIZE": str(LAYOUT_IMAGE_SIZE),
    "WARMUP_LLM": str(WARMUP_LLM).lower(),
    "WARMUP_EMBEDDING": str(WARMUP_EMBEDDING).lower(),
    "WARMUP_OCR": str(WARMUP_OCR).lower(),
    "WARMUP_LAYOUT": str(WARMUP_LAYOUT).lower(),
})

backend_log = open("/content/chatbot_backend.log", "w")
backend_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", str(BACKEND_PORT)],
    cwd=f"{REPO_DIR}/backend",
    env=backend_env,
    stdout=backend_log,
    stderr=subprocess.STDOUT,
)
print("Backend PID:", backend_proc.pid)

health_url = f"http://127.0.0.1:{BACKEND_PORT}/health"
for _ in range(60):
    if backend_proc.poll() is not None:
        backend_log.flush()
        print(Path("/content/chatbot_backend.log").read_text()[-4000:])
        raise RuntimeError(f"Backend exited during startup with code {backend_proc.returncode}")
    try:
        response = requests.get(health_url, timeout=3)
        if response.status_code == 200:
            print(response.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print(Path("/content/chatbot_backend.log").read_text()[-4000:])
    raise RuntimeError("Backend did not become healthy")


Backend PID: 2401
{'status': 'ok', 'document_rag_documents': 0, 'document_rag_chunks': 0, 'pdf_rag_documents': 0, 'pdf_rag_chunks': 0, 'ollama_api_url': 'http://127.0.0.1:11434/api/generate', 'ollama_model': 'hf.co/Zkare/Chatbot_Ielts_Assistant_v2:Q4_K_M', 'ollama_num_predict': 1200}


In [10]:
import json
import requests
from pathlib import Path

warmup_url = f"http://127.0.0.1:{BACKEND_PORT}/warmup"
response = requests.post(warmup_url, timeout=1200)
print("Warmup status:", response.status_code)
try:
    warmup = response.json()
    results = warmup.get("results") or {}
    summary = {name: status.get("ok", status.get("skipped", False)) for name, status in results.items()}
    print("Warmup summary:", json.dumps(summary, ensure_ascii=False))
except Exception:
    print(response.text[:4000])
    warmup = {}

if response.status_code != 200:
    print(Path("/content/chatbot_backend.log").read_text()[-4000:])
    raise RuntimeError("Backend warmup failed")

llm_status = (warmup.get("results") or {}).get("llm") or {}
if WARMUP_LLM and not llm_status.get("ok"):
    print("LLM warmup detail:", json.dumps(llm_status, ensure_ascii=False, indent=2))
    print(Path("/content/chatbot_backend.log").read_text()[-4000:])
    raise RuntimeError(
        "Ollama LLM warmup failed. Do not start evaluation or the tunnel until llm.ok=true."
    )

layout_status = (warmup.get("results") or {}).get("layout") or {}
if WARMUP_LAYOUT and not layout_status.get("ok"):
    print("Layout warmup detail:", json.dumps(layout_status, ensure_ascii=False, indent=2))
    print(Path("/content/chatbot_backend.log").read_text()[-4000:])
    raise RuntimeError(
        "DocLayout-YOLO warmup failed. Check layout warmup detail above for the exact model/device/download error."
    )

ocr_status = (warmup.get("results") or {}).get("ocr") or {}
if WARMUP_OCR and not ocr_status.get("ok"):
    print("OCR warmup detail:", json.dumps(ocr_status, ensure_ascii=False, indent=2))
    print(Path("/content/chatbot_backend.log").read_text()[-4000:])
    raise RuntimeError(
        "RapidOCR warmup failed. Do not upload images/scanned PDFs until ocr.ok=true. "
        "Check OCR warmup detail above for the exact model/config/download error."
    )



Warmup status: 200
Warmup summary: {"llm": true, "embedding": true, "layout": true, "ocr": true}


In [ ]:
# Upload the seven-document corpus and collect 56 answers with full RAG debug data.
!cd {REPO_DIR} && python backend/tools/chat_evaluation.py --base-url http://127.0.0.1:{BACKEND_PORT}


In [ ]:
from google.colab import files
from pathlib import Path

reports = sorted(
    Path(f"{REPO_DIR}/backend/data/chat_evaluation").glob("chat-review-capture-*.json"),
    key=lambda path: path.stat().st_mtime,
)
assert reports, "Không tìm thấy report."
files.download(str(reports[-1]))


In [12]:
!cd {REPO_DIR}/frontend && npm ci --include=optional --no-audit --no-fund


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 163 packages in 8s
⠇npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.1
npm notice To update run: npm install -g npm@12.0.1
npm notice
⠇
> ielts-chatbot-frontend@1.0.0 prebuild
> node scripts/ensure-rollup-native.cjs

Installing missing Rollup native package: @rollup/rollup-linux-x64-gnu@4.62.2
⠙⠹⠸⠼⠴
added 1 package in 752ms
⠴node:internal/modules/cjs/loader:1215
  throw err;
  ^

Error: Cannot find module '@rollup/rollup-linux-x64-gnu'
Require stack:
- /content/ielts-chatbot/frontend/scripts/ensure-rollup-native.cjs
    at Module._resolveFilename (node:internal/modules/cjs/loader:1212:15)
    at Module._load (node:internal/modules/cjs/loader:1043:27)
    at Module.require (node:internal/modules/cjs/loader:1298:19)
    at require (node:internal/modules/helpers:182:18)
    at requireNativePackage (/content/ielts

In [13]:
import os
import subprocess
import time
import requests
from pathlib import Path

subprocess.run(["pkill", "-f", "vite"], check=False)
time.sleep(2)

frontend_env = os.environ.copy()
frontend_env.update({"VITE_CHATBOT_API_URL": "/api"})

frontend_dir = f"{REPO_DIR}/frontend"

frontend_log = open("/content/frontend.log", "w")
frontend_proc = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd=frontend_dir,
    env=frontend_env,
    stdout=frontend_log,
    stderr=subprocess.STDOUT,
)
print("Frontend PID:", frontend_proc.pid)

frontend_url = f"http://127.0.0.1:{FRONTEND_PORT}"
for _ in range(45):
    try:
        response = requests.get(frontend_url, timeout=3)
        if response.status_code < 500:
            print("Frontend status:", response.status_code)
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print(Path("/content/frontend.log").read_text()[-4000:])
    raise RuntimeError("Frontend did not start")


Frontend PID: 3638
Frontend status: 200


In [14]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared


In [ ]:
# Keep this cell running. Open the printed trycloudflare URL.
!/content/cloudflared tunnel --protocol http2 --url http://127.0.0.1:8000
